# GlobIO: inference and MC dropout uncertainty

This notebook loads a trained model, predicts the profiles of a dataset CSV,
estimates the uncertainty with Monte Carlo (MC) dropout and plots predicted
and measured profiles. It also shows how to predict new profiles from arrays.

Set the paths in the next cell. Checkpoints written by `train.py` contain the
variable and the normalization statistics; the reference checkpoints
contain only the weights, so for them set `VARIABLE` and `STATS_PATH` (see the
README) or convert them once with `python -m tools.upgrade_checkpoint`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # not needed after `pip install -e .`

from globio import ProfileDataset, load_checkpoint, predict, predict_mc, pressure_levels, resolve_norm_stats
from globio.constants import UNITS
from globio.metrics import profile_rmse, uncertainty_metrics

CHECKPOINT = ROOT / "pretrained_models" / "CHLA_global_profile.pt"   # model to use
DATA_CSV = ROOT / "data" / "CHLA_datasets" / "test_profile_wise_SEANOE.csv"  # profiles to predict
VARIABLE = None    # "CHLA", "NITRATE" or "BBP700": only for checkpoints without metadata
STATS_PATH = None  # e.g. ROOT / "normalization_stats" / "SEANOE_CHLA.json": only for checkpoints without statistics
REGION = None      # e.g. "mediterranean" to keep only the profiles of a region
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

## Load the model and the data

The dataset must be normalized with the statistics used to train the model:
`resolve_norm_stats` returns those stored in the checkpoint (or `STATS_PATH`).

In [ ]:
model, info = load_checkpoint(CHECKPOINT, VARIABLE, device=DEVICE)
stats = resolve_norm_stats(info, STATS_PATH)
variable = model.variable
pressure = pressure_levels(variable)
print(f"{variable} model, dropout {model.dp_rate}, epoch {info.get('epoch')}")

dataset = ProfileDataset(DATA_CSV, variable, stats, region=REGION)
print(f"{len(dataset)} profiles, target available: {dataset.has_target}")
dataset.metadata().head()

## Deterministic prediction

`predict` runs the model in eval mode (dropout off). The result has one row
per profile and one column per pressure level, in physical units.

In [ ]:
prediction = predict(model, dataset, batch_size=256, device=DEVICE)
rmse = profile_rmse(prediction, dataset.target)
print(f"profile-wise RMSE: median {np.median(rmse):.4f}, mean {rmse.mean():.4f} {UNITS[variable]}")

## MC dropout uncertainty

`predict_mc` predicts every profile `n_samples` times with dropout active and
BatchNorm frozen. The mean of the passes is the prediction and their standard
deviation `std` the (model) uncertainty at each level. The reference experiments use 100
passes. Here only a subset of profiles is used to keep the example fast.

In [ ]:
subset = dataset.subset(np.arange(min(200, len(dataset))))
mc = predict_mc(model, subset, n_samples=100, batch_size=32, device=DEVICE, seed=123)
per_profile, coverage, vertical = uncertainty_metrics(subset.target, mc["mean"], mc["std"], pressure=pressure)
print("median RMS uncertainty:", per_profile.uncertainty_rms.median())
coverage

## Plot predicted and measured profiles

Shaded bands: MC mean +/- 1 and 2 standard deviations. Pressure increases
downwards.

In [ ]:
def plot_profile(i, ax):
    mean, std = mc["mean"][i], mc["std"][i]
    ax.fill_betweenx(pressure, mean - 2 * std, mean + 2 * std, color="tab:blue", alpha=0.15, lw=0, label="±2σ")
    ax.fill_betweenx(pressure, mean - std, mean + std, color="tab:blue", alpha=0.3, lw=0, label="±1σ")
    ax.plot(mean, pressure, color="tab:blue", lw=2, label="GlobIO (MC mean)")
    if subset.has_target:
        ax.plot(subset.target[i], pressure, color="tab:red", lw=2, label="measured")
    ax.set_ylim(pressure[-1], pressure[0])
    ax.set_xlabel(f"{variable} [{UNITS[variable]}]")
    meta = subset.metadata().iloc[i]
    ax.set_title(f"{meta.lat:.1f}°, {meta.lon:.1f}°, {meta.year} ({meta.season})", fontsize=10)

examples = np.linspace(0, len(subset) - 1, 4).astype(int)
fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 5), sharey=True)
for ax, i in zip(np.atleast_1d(axes), examples):
    plot_profile(i, ax)
np.atleast_1d(axes)[0].set_ylabel("pressure [dbar]")
np.atleast_1d(axes)[0].legend(loc="lower right", fontsize=8)
fig.tight_layout()

## Predict new profiles from arrays

Profiles must be on the 200-level grid of the variable (`pressure_levels`):
CHLA and BBP700 0-199 dbar every 1 dbar, NITRATE 0-995 dbar every 5 dbar.
`day_rad_from_date` encodes the date with the convention of the training data.
Here the inputs of an existing profile are reused as an example.

In [ ]:
import datetime as dt
from globio import day_rad_from_date

new = ProfileDataset.from_arrays(
    year=[2024], day_rad=[day_rad_from_date(dt.date(2024, 5, 15))], lat=[40.0], lon=[5.0],
    temp=dataset.temp[:1], psal=dataset.psal[:1], doxy=dataset.doxy[:1],
    variable=variable, norm_stats=stats)
new_mc = predict_mc(model, new, n_samples=100, device=DEVICE)
print("surface value:", new_mc["mean"][0, 0], "+/-", new_mc["std"][0, 0])